# Transaction Foundation Model on Ray — Part 5: Batch embedding extraction

<div align="left">
  <a target="_blank" href="https://console.anyscale.com/template-preview/fintech_transaction_fm"><img src="https://img.shields.io/badge/🚀 Run_on-Anyscale-9hf"></a>&nbsp;
  <a href="https://github.com/anyscale/templates/tree/main/templates/fintech_transaction_fm" role="button"><img src="https://img.shields.io/static/v1?label=&message=View%20On%20GitHub&color=586069&logo=github&labelColor=2f363d"></a>&nbsp;
</div>

**⏱️ Time to complete**: ~5 min

---

Part 4 trained the foundation model. The recurring production job is to run it over every card and store a fresh embedding for downstream models to consume. That's a heterogeneous job — CPU to read the tokens, GPU for the forward pass — and it streams through a single Ray Data pipeline.

In [1]:
import sys, os, json

DEMO_ROOT = os.path.abspath(os.getcwd())
if DEMO_ROOT not in sys.path:
    sys.path.insert(0, DEMO_ROOT)

import numpy as np
import pandas as pd

import ray
ray.init(ignore_reinit_error=True, runtime_env={"working_dir": DEMO_ROOT})

from src.paths import artifact_paths, get_demo_base_dir
from src.scale_config import load_scale

SCALE = "full"                       # same knob as the earlier parts; mini = tiny, CPU-only
cfg = load_scale(SCALE)
paths = artifact_paths(get_demo_base_dir(), SCALE)

2026-07-02 18:21:55,273	INFO worker.py:1814 -- Connecting to existing Ray cluster at address: 10.0.119.117:6379...
2026-07-02 18:21:55,311	INFO worker.py:2003 -- Connected to Ray cluster. View the dashboard at https://session-nridlis6nyy2hi9qv4il5lcj6q.i.anyscaleuserdata.com 
2026-07-02 18:21:55,348	INFO packaging.py:392 -- Ignoring upload to cluster for these files: [PosixPath('/home/ray/default_cld_g54aiirwj1s8t9ktgzikqur41k/templates/templates/fintech_transaction_fm/dot.claude/plugins/marketplaces/claude-plugins-official/.gitignore')]
2026-07-02 18:21:55,441	INFO packaging.py:392 -- Ignoring upload to cluster for these files: [PosixPath('/home/ray/default_cld_g54aiirwj1s8t9ktgzikqur41k/templates/templates/fintech_transaction_fm/dot.claude/plugins/marketplaces/claude-plugins-official/plugins/hookify/.gitignore')]
2026-07-02 18:21:55,592	INFO packaging.py:691 -- Creating a file package for local module '/home/ray/default_cld_g54aiirwj1s8t9ktgzikqur41k/templates/templates/fintech_trans

## Turn the trained model into one vector per window

The foundation model's value isn't its loss — it's the **embedding**: pool the encoder's hidden states into a single vector that summarizes a card's recent behavior, and hand that vector to whatever downstream model needs it (fraud in Part 6, but also churn, credit, personalization). Computing those vectors for every card on a schedule is the job that actually runs in production, over and over, as new transactions arrive.

We pool with `"last"` — the hidden state at the most recent transaction — because the eval windows are labeled per-transaction (is *this* transaction fraud?), and the last position is the readout aligned with that label. (For a whole-card summary you'd mean-pool instead.) Each embedding is written alongside its `label`, `split`, importance `weight`, and the target transaction's raw features, so Part 6 has everything in one place to compare raw-vs-FM-vs-fusion without re-reading anything.

## Embed every window with a pool of model replicas

This is the stage where Ray Data is genuinely differentiated, not just convenient. The forward pass wants a GPU, but the model is expensive to load — so loading it per batch would be hopeless. The fix is a **stateful** `map_batches`: `EmbeddingExtractor` is a callable class that loads the model **once per replica** in `__init__` and embeds each batch in `__call__`.

`ActorPoolStrategy(min_size=1, max_size=num_workers)` runs a pool of those replicas, growing from one up to `num_workers` as work demands; `num_gpus=1` places each on a GPU (and `num_cpus=1` keeps it on CPU at `mini`). The CPU Parquet read and the GPU forward pass run as one streamed pipeline — batches flow from the readers to the actors through the object store with no intermediate disk writes. Going from CPU here to a pool of GPU replicas at `full` is the `embed` config, not a code change; `scripts/04_extract_embeddings.py` runs this same `EmbeddingExtractor`.

In [2]:
from src.embed import EmbeddingExtractor, embedding_health

eb = cfg["embed"]
print(f"embedding with up to {eb['num_workers']} replica(s), "
      f"{'GPU' if eb['use_gpu'] else 'CPU'}, batch_size={eb['batch_size']}")

# Re-runs reuse the cached embeddings.
if not os.path.exists(paths["embeddings"]):
    ds = ray.data.read_parquet(paths["tokenized_eval"])   # CPU read

    # Stateful map_batches: the model loads ONCE per replica (in __init__),
    # then embeds each batch in __call__ — not reloaded per batch.
    embedded = ds.map_batches(
        EmbeddingExtractor,
        fn_constructor_kwargs={"checkpoint_dir": paths["checkpoint"], "pooling": "last"},
        batch_size=eb["batch_size"],
        compute=ray.data.ActorPoolStrategy(min_size=1, max_size=eb["num_workers"]),  # 1..N replicas
        num_gpus=1 if eb["use_gpu"] else 0,                          # one GPU each when use_gpu
        num_cpus=1,                                                  # finite footprint (see Part 5 notes)
        batch_format="numpy",
    )
    embedded.write_parquet(paths["embeddings"])           # streamed write, no intermediate disk
    print(f"  wrote embeddings -> {paths['embeddings']}")
else:
    print(f"  reusing cached embeddings at {paths['embeddings']}")

embedding with up to 16 replica(s), GPU, batch_size=256


2026-07-02 18:22:00,323	INFO logging.py:416 -- Registered dataset logger for dataset dataset_192_0
2026-07-02 18:22:00,347	WARNING map_operator.py:927 -- Specifying both num_cpus and num_gpus for map tasks is experimental, and may result in scheduling or stability issues. Please report any issues to the Ray team: https://github.com/ray-project/ray/issues/new/choose
2026-07-02 18:22:00,349	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_192_0. Full logs are in /tmp/ray/session_2026-07-02_10-58-50_130263_2811/logs/ray-data
2026-07-02 18:22:00,350	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_192_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ListFiles] -> TaskPoolMapOperator[ReadFiles] -> ActorPoolMapOperator[MapBatches(EmbeddingExtractor)] -> TaskPoolMapOperator[Write]
{"asctime":"2026-07-02 18:22:00,389","levelname":"E","message":"Actor with class name: 'MapWorker(MapBatches(EmbeddingExtractor))' and ID: 'bfeeb4d65b04b132ebfe0af216000000'

(autoscaler +9s) Tip: use `ray status` to view detailed cluster status. To disable these messages, set RAY_SCHEDULER_EVENTS=0.
(autoscaler +13s) [autoscaler] [1xL4:4CPU-16GB] Attempting to add 1 node to the cluster (increasing from 0 to 1).
(autoscaler +13s) [autoscaler] [1xL4:4CPU-16GB|g6.xlarge] [us-west-2b] [on-demand] Launched 1 instance.


2026-07-02 18:22:10,632	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_192_0 =======
2026-07-02 18:22:10,633	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-07-02 18:22:10,634	INFO logging_progress.py:227 -- Active & requested resources: 148/184 CPU, 54.5GiB/105.1GiB object store (pending: 1 CPU, 1 GPU)
2026-07-02 18:22:10,634	INFO logging_progress.py:181 -- 
2026-07-02 18:22:10,635	INFO logging_progress.py:231 -- ListFiles: 652/652
2026-07-02 18:22:10,635	INFO logging_progress.py:233 --   Tasks: 0; Actors: 0; Queued blocks: 0 (0.0B); Resources: 0.0 CPU, 72.5KiB object store
2026-07-02 18:22:10,636	INFO logging_progress.py:231 -- ReadFiles: 1238435/5662088
2026-07-02 18:22:10,637	INFO logging_progress.py:233 --   Tasks: 149; Actors: 0; Queued blocks: 26 (10.2KiB); Resources: 149.0 CPU, 54.8GiB object store
2026-07-02 18:22:10,637	INFO logging_progress.py:231 -- MapBatches(EmbeddingExtractor): 0/1
2026-07-02 18:22:10,637	INFO logging_progress.py:233 --   Tasks

(autoscaler +23s) [autoscaler] [64CPU-256GB] Attempting to add 3 nodes to the cluster (increasing from 0 to 3).


...

(autoscaler +8m53s) [autoscaler] Cluster upscaled to {388 CPU, 18 GPU}.


2026-07-02 18:30:54,122	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_192_0 =======
2026-07-02 18:30:54,123	INFO logging_progress.py:225 -- Total Progress: 17/762
2026-07-02 18:30:54,123	INFO logging_progress.py:227 -- Active & requested resources: 14/388 CPU, 14/18 GPU, 80.0GiB/222.1GiB object store (pending: 2 CPU, 2 GPU)
2026-07-02 18:30:54,124	INFO logging_progress.py:181 -- 
2026-07-02 18:30:54,124	INFO logging_progress.py:231 -- ListFiles: 652/652
2026-07-02 18:30:54,125	INFO logging_progress.py:233 --   Tasks: 0; Actors: 0; Queued blocks: 0 (0.0B); Resources: 0.0 CPU, 0.0B object store
2026-07-02 18:30:54,125	INFO logging_progress.py:231 -- ReadFiles: 5198460/5198460
2026-07-02 18:30:54,126	INFO logging_progress.py:233 --   Tasks: 0; Actors: 0; Queued blocks: 0 (0.0B); Resources: 0.0 CPU, 79.8GiB object store
2026-07-02 18:30:54,126	INFO logging_progress.py:231 -- MapBatches(EmbeddingExtractor): 118242/5300024
2026-07-02 18:30:54,127	INFO logging_progress.py:2

(MapWorker(MapBatches(EmbeddingExtractor)) pid=4041, ip=10.0.117.176) [embed] loaded TransactionFM on cuda [repeated 8x across cluster]


(MapWorker(MapBatches(EmbeddingExtractor)) pid=4040, ip=10.0.117.176) /tmp/ray/session_2026-07-02_10-58-50_130263_2811/runtime_resources/working_dir_files/_ray_pkg_4d898a354b8ecd70/src/embed.py:57: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
(MapWorker(MapBatches(EmbeddingExtractor)) pid=4040, ip=10.0.117.176)   return torch.as_tensor(v, dtype=dtype, device=self.device)
2026-07-02 18:32:04,305	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_192_0 =======
2026-07-02 18:32:04,306	INFO logging_progress.py:225 -- Total Progress: 32/762
2026-07-02 18:32:04,306	INFO logging_progress.py:227 -- Active & req

  wrote embeddings -> /mnt/cluster_storage/transaction-fm/embeddings/full/


## Did the embeddings actually learn anything?

The classic self-supervised failure is silent **representation collapse**: every customer maps to nearly the same vector while the loss looks fine. A cheap guard is to sample the embeddings and check two numbers — mean pairwise cosine similarity (→1.0 means collapse) and mean feature variance (→0 means collapse). `embedding_health` reports both.

In [3]:
embedding_health(paths["embeddings"])

# Don't pull the whole table into pandas just to peek at it — at `full` that's
# millions of rows, each carrying a tensor column that pandas materializes as a
# separate numpy object. Read the count from Parquet metadata and grab one row.
eds = ray.data.read_parquet(paths["embeddings"])
n_windows = eds.count()                            # metadata only, no vectors read
row = eds.limit(1).to_pandas().iloc[0]             # a single window for the example
vec0 = np.asarray(row["embedding"])                # Ray stores this as a tensor column
carried = [c for c in row.index if c != "embedding"]

print(f"\n{n_windows:,} window embeddings · dim {vec0.shape[-1]}")
print(f"carried alongside each vector (for Part 6): {carried}")

print(f"\nexample — card {int(row['card_id'])}, label {int(row['label'])}, split {row['split']}")
print(f"  embedding[:8] = {vec0[:8].round(3).tolist()}")

2026-07-02 19:19:23,967	INFO logging.py:416 -- Registered dataset logger for dataset dataset_195_0
2026-07-02 19:19:23,972	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_195_0. Full logs are in /tmp/ray/session_2026-07-02_10-58-50_130263_2811/logs/ray-data
2026-07-02 19:19:23,972	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_195_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ListFiles] -> TaskPoolMapOperator[ReadFiles] -> LimitOperator[limit=2000]
2026-07-02 19:19:23,991	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_195_0 =======
2026-07-02 19:19:23,992	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-07-02 19:19:23,993	INFO logging_progress.py:227 -- Active & requested resources: 0/388 CPU, 0.0B/222.1GiB object store
2026-07-02 19:19:23,994	INFO logging_progress.py:181 -- 
2026-07-02 19:19:23,994	INFO logging_progress.py:231 -- ListFiles: 0/1
2026-07-02 19:19:23,995	INFO logging_progress.py:233 --   Tasks: 1; 

[embed] health: mean_pairwise_cos=0.089 (→1 = collapse), mean_feature_var=5.8876, n=2000


2026-07-02 19:19:27,287	INFO streaming_executor.py:294 -- ✔️  Dataset dataset_197_0 execution finished in 2.08 seconds
INFO:openlineage.client.client:OpenLineageClient will use `composite` transport
INFO:openlineage.client.transport.composite:Stopping OpenLineage CompositeTransport emission after the first successful delivery because `continue_on_success=False`. Transport that emitted the event: <HttpTransport(name=first, kind=http, priority=1)>
2026-07-02 19:19:27,302	INFO logging.py:416 -- Registered dataset logger for dataset dataset_198_0
2026-07-02 19:19:27,306	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_198_0. Full logs are in /tmp/ray/session_2026-07-02_10-58-50_130263_2811/logs/ray-data
2026-07-02 19:19:27,307	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_198_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ListFiles] -> TaskPoolMapOperator[ReadFiles] -> LimitOperator[limit=1]
2026-07-02 19:19:27,325	INFO logging_progress.py:174


5,198,460 window embeddings · dim 512
carried alongside each vector (for Part 6): ['card_id', 'label', 'split', 'weight', 'raw_amount', 'raw_hour', 'raw_dow', 'raw_mcc', 'raw_ts', 'raw_use_chip', 'raw_merchant_state', 'raw_merchant_city', 'raw_zip', 'raw_merchant_id', 'raw_card_id']

example — card 112903, label 0, split val
  embedding[:8] = [0.4399999976158142, 1.8559999465942383, -0.5519999861717224, 3.7109999656677246, 0.20800000429153442, -5.059000015258789, -2.436000108718872, -0.050999999046325684]


## Takeaways

**Ray Data**
- A stateful `map_batches` (a callable class + `ActorPoolStrategy`) loads the model **once per replica**, not once per batch — the difference between usable and hopeless for GPU inference.
- `num_gpus` on `map_batches` places each replica on a GPU; the CPU read and GPU forward pass run as one streamed pipeline with no intermediate disk. This heterogeneous CPU→GPU shape is where Ray Data genuinely earns its keep.
- The same code goes from one CPU replica at `mini` to a pool of GPU replicas at `full` by changing the `embed` config; the notebook runs the same `EmbeddingExtractor` as `scripts/04_extract_embeddings.py`.

**The embeddings**
- One pooled vector per window (`"last"` pooling = the most recent transaction's hidden state, aligned with the per-transaction fraud label).
- Each vector is written next to its `label`, `split`, `weight`, and raw target features, so the downstream stage needs no re-reads.
- A quick collapse check (mean pairwise cosine, feature variance) catches the silent self-supervised failure mode before it reaches the downstream model.

---

## Next

**Part 6 — Downstream fraud: raw vs FM vs fusion**: train the same XGBoost recipe on three feature sets — raw transaction fields, the FM embedding, and the two fused — and measure the lift at natural fraud prevalence with PR-AUC.